In [44]:
#词向量
import pandas as pd
import jieba
from gensim.models.word2vec import Word2Vec

# 读入训练集文件
data = pd.read_csv('train.csv')
# 转字符串数组
corpus = data['comment'].values.astype(str)
# 分词，再重组为字符串数组
corpus = [jieba.lcut(corpus[index]
                          .replace("，", "")
                          .replace("!", "")
                          .replace("！", "")
                          .replace("。", "")
                          .replace("~", "")
                          .replace("；", "")
                          .replace("？", "")
                          .replace("?", "")
                          .replace("【", "")
                          .replace("】", "")
                          .replace("#", "")
                        ) for index in range(len(corpus))]
# 词向量模型训练
model = Word2Vec(corpus, sg=0, vector_size=300, window=7, min_count=5, workers=4)
#模型显示
print('模型参数：',model,'\n')
#最匹配
print('最匹配的词是：',model.wv.most_similar(positive=['点赞', '不错'], negative=['难吃']),'\n')
#最不匹配
#print('最不匹配的词是：',model.wv.doesnt_match("点赞 好吃 支持 难吃".split()),'\n')
#语义相似度
print('相似度为=',model.wv.similarity('推荐','好吃'),'\n')
#坐标返回
print(model.wv.__getitem__('地道'))

模型参数： Word2Vec<vocab=2762, vector_size=300, alpha=0.025> 

最匹配的词是： [('位置', 0.956965982913971), ('好找', 0.9479045867919922), ('热情', 0.9356024265289307), ('推荐', 0.9340485334396362), ('实惠', 0.9331411123275757), ('老板娘', 0.931714653968811), ('可以', 0.9293155074119568), ('适合', 0.9293134808540344), ('高', 0.9285231232643127), ('值得', 0.9244087934494019)] 

相似度为= 0.8583281 

[ 3.10859252e-02  4.16599363e-02  3.40370089e-02  7.17792362e-02
 -5.04505672e-02 -1.20197169e-01  6.77129999e-02  2.59585440e-01
  4.94909883e-02 -2.48822104e-02  2.48736627e-02 -1.13429286e-01
 -3.93153541e-02  1.28208613e-02 -1.24398567e-01 -5.52913286e-02
  1.19400300e-01  2.01937854e-02  5.89785129e-02 -9.58669484e-02
 -6.32174090e-02 -3.64807667e-03  1.49264680e-02  1.19875483e-02
  6.58756495e-02 -4.98809107e-02 -1.80376649e-01  2.85810605e-02
 -2.26929504e-02 -1.05924182e-01  5.36112152e-02 -3.41093428e-02
  3.30865234e-02  6.96513578e-02 -9.15797129e-02  2.82894000e-02
  5.74943870e-02 -1.10405482e-01 -8.62193119e-05 

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [34]:
from gensim.models.word2vec import Word2Vec

# 假设 corpus 是你已经预处理好的分词列表
model_skipgram = Word2Vec(
    corpus,
    sg=1,          # 1=Skip-Gram, 0=CBOW
    vector_size=300,
    window=5,
    min_count=3,
    workers=4
)
print("Skip-Gram模型参数：", model_skipgram)
print("词汇表大小：", len(model_skipgram.wv.key_to_index))

Skip-Gram模型参数： Word2Vec<vocab=4036, vector_size=300, alpha=0.025>
词汇表大小： 4036


In [35]:
env_vec = model_skipgram.wv['环境']
print("“环境”词向量（前20维）：")
print(env_vec[:20])
print("\n词向量形状：", env_vec.shape)

“环境”词向量（前20维）：
[ 0.09914041  0.08458868 -0.04788452  0.09143654 -0.09329107 -0.30925176
 -0.08619627  0.5047873  -0.2754182  -0.10909923  0.14302425 -0.30064684
 -0.25243312  0.07330522 -0.13413237  0.03858657  0.36852947  0.11112601
  0.29746    -0.43049747]

词向量形状： (300,)


In [36]:
similar_words = model_skipgram.wv.most_similar('好吃', topn=3)
print("与“好吃”最接近的3个词：")
for word, sim in similar_words:
    print(f"{word}：{sim:.4f}")

与“好吃”最接近的3个词：
入味：0.8318
好看：0.8311
棒：0.8224


In [41]:
sim_good_delicious = model_skipgram.wv.similarity('好吃', '美味')
try:
    sim_good_cockroach = model_skipgram.wv.similarity('好吃', '蟑螂')
except KeyError:
    sim_good_cockroach = "“蟑螂”不在词汇表中"

print(f"“好吃”与“美味”的相似度：{sim_good_delicious:.4f}")
print(f"“好吃”与“蟑螂”的相似度：{sim_good_cockroach}")

“好吃”与“美味”的相似度：0.7822
“好吃”与“蟑螂”的相似度：0.31817129254341125


In [45]:
result = model_skipgram.wv.most_similar(
    positive=['餐厅', '聚会'],
    negative=['安静'],
    topn=3
)
print(f"餐厅 + 聚会 - 安静= {result[0][0]}")
print(f"相似度：{result[0][1]:.4f}")

餐厅 + 聚会 - 安静= 窗边
相似度：0.9544


In [46]:
result = model_skipgram.wv.most_similar(
    positive=['餐厅', '聚会'],
    negative=['安静'],
    topn=5
)

print("向量运算「餐厅 + 聚会 - 安静」的前5个最相关结果：")
# 循环输出全部5个结果
for i, (word, sim) in enumerate(result, 1):
    print(f"{i}. {word}，相似度：{sim:.4f}")

向量运算「餐厅 + 聚会 - 安静」的前5个最相关结果：
1. 窗边，相似度：0.9544
2. 家庭聚会，相似度：0.9526
3. 分享，相似度：0.9513
4. 国庆，相似度：0.9442
5. 部门，相似度：0.9425
